# Datamine DECLUST Process Example

This notebook demonstrates how to configure and run the **`declust`** process wrapper in `dmstudio`.

## Process Description

## DECLUST

**Note** : This is a _superprocess_ and running it may have an effect on other Datamine files in the project.

The **DECLUST** process declusters a set of sample data to reduce areas of high-density points. This can be useful where a more uniform distribution of points is required, such as when reconstructing a point cloud or adjusting sample density prior to estimation.

It is common practice to take more samples in high grade areas in order to improve the level of confidence in the estimation. However, when data are not on a regular grid then using the full data set will give a biased estimate of the mean, variance and histogram. In addition, clustered data can affect an indicator variogram.

**Note** : This process supports **[retrieval criteria](<../COMMON/Retrieval_Criteria_Overview.md>)**.

Declustering is the process of adjusting the full data set i.e. by removing or weighting data points in densely sampled areas, to give a more representative and evenly spaced set of samples. The **DECLUST** process provides two methods of doing this, with both methods requiring a regular 3D grid to be placed over the sample data. The two methods are:

1. Sample Selection - select a single sample from each grid cell.
2. Declustered Weight - assign every sample a weight based on the number of samples in the grid cell

If the Sample Selection method is chosen then **DECLUST** provides a choice of four ways of assigning the value to the grid cell:

1. Select a sample at random within each grid cell. A new random sample is generated for each run.
2. Select a sample at random within each grid cell. The same 'random' sample is generated for each run.
3. Select the sample nearest to the grid cell centre.
4. Calculate the average value of all samples within the grid cell.

If the Declustered Weight method is chosen then the weight, **DCWEIGHT** , for a sample is calculated as:

## `DCWEIGHT = NDATA / NCELLS / NPERCELL`

* **where**:

* **NDATA** is the total number of samples

* **NCELLS** is the number of grid cells containing one or more samples

* **NPERCELL** is the number of samples in the grid cell

The sum of the weights over all samples equals the total number of samples (**NDATA**). Therefore if a sample lies in a high density area it will have a weight of less than 1, and if it is in a low density area it will have a weight of more than 1. The output file from the Declustered Weight method can be used to transform data into a normal distribution, for input to the NSCORE or SGSIM processes.

One of the problems with the declustering method is that different grid sizes will generate different statistics. However, in general a regular grid about the size of the average sample spacing is suggested.

The process writes a summary table for each method to the [Output](<../COMMON/Output%20Control%20Bar%20Overview.md>) control bar.

1. **Sample Selection** : shows statistical parameters for each numeric field in the output file. These statistics can be saved to a file if the **STAT_TBL** output file is specified.

2. **Declustered Weigh** t: shows the declustered weight as a function of the number of samples per grid cell. These statistics can be saved to a file if the **WGTS_TBL** output file is specified.

**Note** : The process **DECLUST** has a limit of only being able to use input sample files with a maximum of 53 fields. If the process is run with an input sample file of greater than 53 fields, a warning message is displayed which prompts you to reduce the number of fields.

### Input Files:

* **in** (Undefined):
  Input sample data file. This must contain a set of 3D coordinates (eg X,Y,Z) and at
  least one other field.
  Required=Yes

### Output Files:

* **out** (Undefined):
  Output file containing declustered samples. At least one of the two output files OUT or
  WTOUT must be selected.
  Required=No

* **wtout** (Undefined):
  Output file containing declustered weights. This will be a copy of the IN file, but
  will also include the field DCWEIGHT. At least one of the two output files OUT or WTOUT
  must be selected.
  Required=No

* **wgts_tbl** (Undefined):
  Output file containing summary statistics for declustered weights.
  Required=No

* **stat_tbl** (Undefined):
  Output file containing summary statistics for declustered and clustered WTFIELD
  samples.
  Required=No

### Fields:

* **x** (Numeric : IN):
  X coordinate of sample data
  Default=X
  Required=Yes

* **y** (Numeric : IN):
  Y coordinate of sample data
  Default=Y
  Required=Yes

* **z** (Numeric : IN):
  Z coordinate of sample data
  Default=Z
  Required=Yes

* **wtfield** (Numeric : IN):
  Field to be used for calculating declustered weights. This is only relevant if a WTOUT
  file has been specified and one or more of the grade fields in the IN file contain
  absent data values. Specifying a WTFIELD field ensures that records containing absent
  data values for that field will be ignored. If a WTFIELD field is not specified but a
  WTOUT file has been selected then the Z field is used.
  Default=Undefined
  Required=No

### Parameters:

* **method**:
  Declustering method if OUT file specified: 1 = random selection within grid (different
  selection each run) 2 = pseudo random selection within grid (repeatable) 3 = nearest to
  grid centre 4 = average of samples within grid
  Range=1,4
  Values=1,2,3,4
  Default=1
  Required=No

* **xgrid**:
  Grid size in X
  Range=0.00001,+
  Values=Undefined
  Default=Undefined
  Required=Yes

* **ygrid**:
  Grid size in Y
  Range=0.00001,+
  Values=Undefined
  Default=Undefined
  Required=Yes

* **zgrid**:
  Grid size in Z
  Range=0.00001,+
  Values=Undefined
  Default=Undefined
  Required=Yes

* **xorig**:
  X coordinate of grid origin
  Range=Undefined
  Values=Undefined
  Default=0
  Required=No

* **yorig**:
  Y coordinate of grid origin
  Range=Undefined
  Values=Undefined
  Default=0
  Required=No

* **zorig**:
  Z coordinate of grid origin
  Range=Undefined
  Values=Undefined
  Default=0
  Required=No

* **centre**:
  Flag to show whether the X, Y and Z coordinates of the grid centre should be included
  in the OUT file. If selected the names of the fields in the file will be XCENTRE,
  YCENTRE and ZCENTRE:
  Range=0,1
  Values=0,1
  Default=0
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Initialize Sandbox
import os
import shutil
import glob
import pandas as pd
from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Initialize active project sandbox using the shared test_sandbox project
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()
agent.initialize_sandbox(notebook_folder)

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('declust')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine database locally to this sandbox folder using a `t_` prefix.

In [ ]:
# Step 3: Copy VBOP datasets dynamically from Database to test_sandbox
files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

agent.initialize_sandbox(notebook_folder, files_to_copy=files_to_copy)

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute declust
print("Running declust...")
dm_cmd.declust(
    in_i='t_assays',  # required
    wgts_tbl_o='t_declust_out',  # required
    stat_tbl_o='t_declust_out',  # required
    x_f='X',  # required
    y_f='Y',  # required
    z_f='Z',  # required
    method_p=1,  # required
    xgrid_p='required_val',  # required
    ygrid_p='required_val',  # required
    zgrid_p='required_val',  # required
    # out_o='t_declust_out',  # optional
    # wtout_o='t_declust_out',  # optional
    # wtfield_f='optional',  # optional
    # xorig_p=0,  # optional
    # yorig_p=0,  # optional
    # zorig_p=0,  # optional
    # centre_p=0,  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("declust execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = 't_declust_out.dmx'
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob("t_*.*")
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    if os.path.exists(f):
        try:
            os.remove(f)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = '__pycache__'
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")